# GridFront PPE Model Training

Train a custom YOLOv8 model for construction safety detection on your OAK-D camera.

**Runtime → Change runtime type → T4 GPU** (important!)

## Overview
1. Install dependencies
2. Download a pre-labeled PPE dataset (or upload your own)
3. Fine-tune YOLOv8n on the data
4. Export to ONNX
5. Download weights → convert to blob → deploy on OAK-D

In [ ]:
# Step 1: Install
!pip install ultralytics roboflow -q

In [ ]:
# Verify GPU is available
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE - change runtime!'}")
print(f"CUDA: {torch.cuda.is_available()}")

## Step 2: Get Training Data

**Option A** — Use the Roboflow Construction Safety dataset (quick start, 2,800 images):

**Option B** — Upload your own labeled data (from Roboflow labeling tool)

Run ONE of the cells below.

In [ ]:
# Option A: Download the Roboflow Construction Safety dataset
# This has ~2,800 labeled images with: Hardhat, Mask, NO-Hardhat, NO-Mask,
# NO-Safety Vest, Person, Safety Cone, Safety Vest, machinery, vehicle
#
# 1. Go to https://app.roboflow.com and create a free account
# 2. Go to: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety
# 3. Click 'Download Dataset' → YOLOv8 format → get your API key
# 4. Paste your API key below:

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY_HERE")  # <-- paste your key
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print(f"Dataset downloaded to: {dataset.location}")
print(f"data.yaml: {DATA_YAML}")

In [ ]:
# Option B: Upload your own dataset
# If you labeled your own images in Roboflow and exported as YOLOv8 ZIP:
#
# 1. Upload the ZIP to Colab (drag into the file panel on the left)
# 2. Uncomment and run:

# !unzip your_dataset.zip -d /content/my_dataset
# DATA_YAML = "/content/my_dataset/data.yaml"
# print(f"data.yaml: {DATA_YAML}")

In [ ]:
# Step 3: Train!
# Starting from yolov8n (nano) — best speed/accuracy for OAK-D Myriad X
#
# This takes ~20-40 minutes on a T4 GPU with 2,800 images.
# Watch the mAP50 metric — 0.7+ is good, 0.85+ is great.

from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Start from COCO pretrained weights

results = model.train(
    data=DATA_YAML,
    epochs=100,          # More epochs = better (diminishing returns after ~150)
    imgsz=640,           # Match OAK-D input size
    batch=16,            # T4 can handle 16 at 640px
    name="gridfront-ppe",
    patience=20,         # Stop early if no improvement for 20 epochs
    save=True,
    plots=True,
)

In [ ]:
# Step 4: Check results
from IPython.display import Image, display
import glob

# Show training curves
results_dir = "runs/detect/gridfront-ppe"
for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    img_path = f"{results_dir}/{img_name}"
    try:
        display(Image(filename=img_path, width=800))
        print(f"\n{img_name}")
    except:
        pass

In [ ]:
# Step 5: Export to ONNX (needed for blob conversion)
best_model = YOLO("runs/detect/gridfront-ppe/weights/best.pt")
best_model.export(format="onnx", imgsz=640, simplify=True)
print("\nExported to ONNX!")
print("File: runs/detect/gridfront-ppe/weights/best.onnx")

In [ ]:
# Step 6: Download your trained model files
# Download BOTH files — you'll need the .onnx for blob conversion

from google.colab import files

files.download("runs/detect/gridfront-ppe/weights/best.pt")
files.download("runs/detect/gridfront-ppe/weights/best.onnx")
print("\nDownloaded! Next step: convert .onnx to .blob at https://tools.luxonis.com")

## Step 6: Convert to OAK-D blob

1. Go to **https://tools.luxonis.com**
2. Click **ONNX to blob**
3. Upload `best.onnx`
4. Settings:
   - **OpenVINO version**: 2022.1 (for RVC2/Myriad X)
   - **Shaves**: 6
   - **Data type**: FP16
5. Download the `.blob` file
6. Copy it to your project: `models/gridfront-ppe-v1.blob`

## Step 7: Deploy

In `pipeline/model_registry.py`, add:

```python
_register(ModelDef(
    id="gridfront-ppe-v1",
    name="GridFront PPE v1 (Custom)",
    slug="",
    classes=PPE_LABELS,
    input_size=(640, 640),
    description="Custom trained PPE model for your site.",
    source="local",
    blob_path="models/gridfront-ppe-v1.blob",
))
```

Then select it in Settings → Models and restart the server.